In [30]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

In [31]:
big_path = "Data/big_data.csv"
smaller_path = "Data/smaller_data.csv"

df = pd.read_csv(smaller_path)

df['label'] = df['label'].map({0: 'real', 1: 'fake'})

# 1. Fill any NaNs that appeared during the heavy cleaning
df['text'] = df['text'].fillna("")

# 2. (Optional but recommended) Remove rows that became empty strings 
# after masking/cleaning so they don't confuse the model
df = df[df['text'].str.strip() != ""]

# Now proceed to your split
# X_train, X_test, y_train, y_test = train_test_split(df['text'], df['label'], ...)

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42
)

In [32]:
custom_stops = list(ENGLISH_STOP_WORDS) + [
    'monday', 'tuesday', 'wednesday', 'thursday', 'friday', 'saturday', 'sunday',
    'text', 'related', 'coverage', 'told', 'said', 'euros'
]

vectorizer = TfidfVectorizer(stop_words=custom_stops, max_df=0.7, min_df=5,ngram_range=(1,2))

model = make_pipeline(vectorizer, LogisticRegression())
model.fit(X_train, y_train)

predictions = model.predict(X_test)

print(classification_report(y_test, y_pred=predictions))
score = model.score(X_test, y_test)
print(f"Model score: {score}")

ValueError: np.nan is an invalid document, expected byte or unicode string.

In [25]:
def predict_article(text):
    return model.predict([text])[0]

real_text = "Popular streamer Clavicular framemogged by ASU frat leader."
prediction = predict_article(real_text)

print(f"The model predicts the real text as: {prediction}")

fake_text = "Scientists discover that the moon is made out of cheese."
prediction = predict_article(fake_text)

print(f"The model predicts the fake text as: {prediction}")

The model predicts the real text as: real
The model predicts the fake text as: real


In [26]:
import pandas as pd

def print_top_features(vectorizer, model, n_top=20):
    features = vectorizer.get_feature_names_out()
    coefs = model.coef_[0]
    
    df = pd.DataFrame({'word': features, 'weight': coefs})
    
    print("--- Top Predictors for FAKE (Class 1) ---")
    print(df.sort_values(by='weight', ascending=False).head(n_top).to_string(index=False))
    
    print("\n--- Top Predictors for REAL (Class 0) ---")
    print(df.sort_values(by='weight', ascending=True).head(n_top).to_string(index=False))

# Example usage assuming your fitted models are named 'tfidf' and 'log_model'
# 1. Extract the fitted vectorizer and model from your pipeline
fitted_vectorizer = model[0]
fitted_log_reg = model[1]

# 2. Call the function using those extracted pieces
print_top_features(fitted_vectorizer, fitted_log_reg)

--- Top Predictors for FAKE (Class 1) ---
      word   weight
    trumps 3.238085
    source 3.061054
 reporters 2.550223
 officials 2.298973
   matters 2.168607
     sales 1.951199
      week 1.934748
   sources 1.874905
   earlier 1.851596
   britain 1.833779
republican 1.830928
   british 1.819145
   details 1.797830
    versus 1.752824
      late 1.752007
previously 1.733180
   million 1.720546
   despite 1.705280
opposition 1.673179
    sports 1.629385

--- Top Predictors for REAL (Class 0) ---
     word    weight
    going -4.915623
according -4.706655
     able -4.502221
    think -4.467710
statement -4.232643
    claim -3.692750
 expected -3.637673
 continue -3.521478
     know -3.444992
     time -3.307622
     dont -3.279087
      end -3.221160
    photo -3.215336
  working -3.204387
    image -2.991870
     sure -2.989177
   number -2.983105
     year -2.873051
   ensure -2.828717
   people -2.803780
